# 6. Clustering

Clustering groups similar conformations. This chapter runs GENESIS's built-in
**k-means** (`kmeans_clustering`) on the bundled BPTI trajectory, then reproduces
the idea with **scikit-learn** on the same coordinates — illustrating how genepie
results flow into the ML ecosystem.


In [ ]:
import os, tempfile
import numpy as np
from genepie import genesis_exe, SMolecule
from genepie.tests.conftest import BPTI_PDB, BPTI_PSF, BPTI_DCD

# kmeans_clustering writes cluster PDBs to the working directory; run in a temp dir.
os.chdir(tempfile.mkdtemp())

mol = SMolecule.from_file(pdb=BPTI_PDB, psf=BPTI_PSF)
trajs, _ = genesis_exe.crd_convert(
    mol, trj_files=[str(BPTI_DCD)], trj_format="DCD",
    trj_type="COOR+BOX", selection="all",
)
traj = trajs[0]

## GENESIS k-means

Cluster on the C&alpha; atoms after TR+ROT fitting.

In [ ]:
result = genesis_exe.kmeans_clustering(
    mol, traj,
    selection_group=["an:CA"],
    fitting_method="TR+ROT", fitting_atom=1,
    analysis_atom=1, num_clusters=2,
    max_iteration=100, stop_threshold=98.0, num_iterations=5,
    trjout_atom=1, trjout_format="DCD", trjout_type="COOR",
    check_only=False, allow_backup=False, iseed=3141592,
)
genesis_labels = np.asarray(result.cluster_idxs)
print("per-frame cluster indices:", genesis_labels)
print("representative structures :", len(result.mols_from_pdb))

## scikit-learn on the same coordinates

Flatten each fitted C&alpha; frame into a feature vector and run
`sklearn.cluster.KMeans`. (Cluster *labels* are arbitrary integers, so we compare
the *partition* rather than exact label values.)


In [ ]:
from sklearn.cluster import KMeans

ca_trajs, _ = genesis_exe.crd_convert(
    mol, trj_files=[str(BPTI_DCD)], trj_format="DCD",
    trj_type="COOR+BOX", selection="an:CA",
    fitting_selection="an:CA", fitting_method="TR+ROT",
)
X = ca_trajs[0].coords.reshape(ca_trajs[0].nframe, -1)   # (nframe, 3*n_ca)
sk_labels = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(X)
print("sklearn labels :", sk_labels)
print("genesis labels :", genesis_labels)

## Visualize the two label assignments

In [ ]:
import plotly.io as pio
import plotly.graph_objects as go
pio.renderers.default = "notebook"

frames = np.arange(len(genesis_labels))
fig = go.Figure()
fig.add_trace(go.Scatter(x=frames, y=genesis_labels, mode="markers",
                         name="GENESIS k-means", marker=dict(size=12, symbol="circle")))
fig.add_trace(go.Scatter(x=frames, y=sk_labels + 0.1, mode="markers",
                         name="scikit-learn", marker=dict(size=12, symbol="x")))
fig.update_layout(title="Cluster assignment per frame", xaxis_title="Frame",
                  yaxis_title="cluster id", template="plotly_white", height=340,
                  yaxis=dict(tickvals=[0, 1]))
fig